# Invariance — invariance of a uniform magnetic field

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``invariance``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

With $b\equiv b_0$ and $u=0$ both transport terms vanish identically, so
$b(x,t)$ must stay equal to $b_0$ for all $t$. Machine-precision invariant:
it catches spurious sources, a wrong time factor or bad CL assembly.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 5 | no spurious source (equilibria preserved) | exact invariant |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset carries the
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks that `trust -partition`
toggles to build `PAR_invariance.data` + the `.Zones` files; the `# BEGIN MESH #`
wrapper hides the mesh generation in the parallel run (upstream `grad_rot.data` pattern).


In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "invariance.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition invariance.data 2 -> PAR_invariance.data + DOM_000*.Zones
# --- mesh-convergence scan (shared ladder: L1 = cells x2, L2 = cells x4) ---
# Levels instantiated from src/invariance.tpl.data (byte-copy of the baseline
# dataset with $nx/$ny node-count placeholders); the baseline case above is
# the untouched L0. Each level runs in its own build/scan/<lvl>/ directory
# so that the partitioned DOM_000*.Zones never overwrite another's.
NX0 = NY0 = 100                       # baseline node counts (L0)
LEVELS = {
    "L1": {"nx": 2*NX0-1, "ny": 2*NY0-1},   # cells x2 -> h/2
    "L2": {"nx": 4*NX0-3, "ny": 4*NY0-3},   # cells x4 -> h/4
}
scan_cases = {}
for lvl, dic in LEVELS.items():
    c = run.addCaseFromTemplate("invariance.tpl.data", f"scan/{lvl}", dic=dic,
                                targetData=f"invariance_{lvl}.data",
                                nbProcs=NB_PROCS, excluNR=True)
    c.partition()          # trust -partition invariance_L1.data 2 -> PAR_*.data + Zones
    scan_cases[lvl] = c
run.printCases()
run.runCases()            # trust PAR_invariance 2

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)

## Checks


In [ ]:
t, b = read_son(case.dataFileName_, "SONDE_CBX")   # -> PAR_invariance_SONDE_CBX.son

b0 = 3.0
err = np.max(np.abs(b - b0)) / abs(b0)
print(f"final time       : {t[-1]:.6g}")
print(f"max |b - b0|/b0 : {err:.3e}")
assert err < 1e-10, f"uniform field not conserved (rel. error {err:.3e})"
print("OK: uniform magnetic field is invariant.")


## Mesh convergence

The shared refinement ladder: `L0` = the baseline mesh above (`100×100`),
`L1` = cells ×2 (`199×199`), `L2` = cells ×4 (`397×397`).

Conservation of a uniform field is a **discrete identity**: with vanishing
transport terms the scheme must reproduce the constant mode exactly, at any
mesh. The scan therefore asserts the invariance bound
$\max_t |b - b_0| / |b_0| < 10^{-10}$ **at every level** — no convergence
order is reported, the identity has no spatial error to converge.

In [ ]:
SONDE_TAG_SCAN = "SONDE_CBX"
B0 = 3.0

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def invariance_error(subdir, case_name):
    """max |b(t) - b0| / |b0| over all recorded times at the scan probe."""
    _, b = read_son_dir(subdir, case_name, SONDE_TAG_SCAN)
    return float(np.max(np.abs(b - B0)) / abs(B0))

scan = {"L0": ("", case.dataFileName_),                       # baseline, build root
        "L1": (f"scan/L1", scan_cases["L1"].dataFileName_),
        "L2": (f"scan/L2", scan_cases["L2"].dataFileName_)}
scan_nodes = {"L0": (100, 100), "L1": (199, 199), "L2": (397, 397)}
scan_err = {lvl: invariance_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes        max |b - b0|/b0")
for lvl in ("L0", "L1", "L2"):
    nx, ny = scan_nodes[lvl]
    print(f"{lvl:<6} {nx:>4}x{ny:<4} {scan_err[lvl]:.3e}")
    assert scan_err[lvl] < 1e-10, \
        f"uniform field not conserved at {lvl} (rel. error {scan_err[lvl]:.3e})"
print("OK: the uniform-field invariance identity holds at every mesh level.")

## Computer Performance


In [ ]:
run.tablePerf()